# 不画回像素，只预测未来特征

JEPA 改变的是预测目标。我们仍看连续视频，但不要求模型猜中下一帧每个像素。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.data import make_pixelworld_dataset
from hwm.jepa import TinyVideoJEPA, patchify_video, jepa_batch_from_episodes, feature_spread
torch.manual_seed(0)


## 1. 从 patch 到短视频块

图片被切成 patch；视频再多一个时间维度。三帧 `16×16` 图片，每帧会得到 16 个 `4×4` patch。

In [ ]:
episodes = make_pixelworld_dataset(6, 6, seed=0)
video, actions, positions = jepa_batch_from_episodes(episodes, history_length=3)
patches = patchify_video(video[:2], patch_size=4)
print('video:', tuple(video.shape), 'patches:', tuple(patches.shape))
assert patches.shape[1:3] == (3, 16)


## 2. Online encoder 与 target encoder 分工

Online encoder 从历史提取信息，Predictor 预测下一帧特征。Target encoder 只交出训练目标，不接收梯度。

In [ ]:
model = TinyVideoJEPA(feature_size=16)
loss, prediction, target, features = model.loss(video, actions=None)
print('prediction/target:', tuple(prediction.shape), tuple(target.shape))
print('target requires_grad:', target.requires_grad)
assert not target.requires_grad


## 3. EMA 不是第二个优化器

Target encoder 缓慢跟随 online encoder。若两边一起被同一个 loss 快速推动，模型更容易找到所有输入都输出同一个常量的捷径。

In [ ]:
parameters = list(model.online_encoder.parameters()) + list(model.predictor.parameters())
optimizer = torch.optim.Adam(parameters, lr=3e-3)
losses, spreads = [], []
for _ in range(35):
    optimizer.zero_grad(); loss, prediction, target, features = model.loss(video, actions=None); loss.backward(); optimizer.step(); model.update_target(momentum=0.99)
    losses.append(float(loss.detach())); spreads.append(float(feature_spread(features).detach()))
print('feature loss:', round(losses[0], 3), '→', round(losses[-1], 3))
print('feature spread:', round(spreads[-1], 3))
assert losses[-1] < losses[0] and spreads[-1] > 0


## 4. Mask 只选择要预测的位置

真正的视频 JEPA 会遮住时空区域。这里用一个棋盘 mask 只计算一半 patch 的误差，先看清接口。

In [ ]:
mask = torch.tensor(([1, 0] * 8), dtype=torch.bool)[None].expand(len(video), -1)
masked_loss, _, _, _ = model.loss(video, mask=mask)
print('masked positions:', int(mask.sum()), 'masked loss:', round(float(masked_loss.detach()), 4))
assert torch.isfinite(masked_loss)


## 小结

到这里我们只证明特征预测能训练，并初步检查没有全变成常量。第二份 Notebook 会问更关键的问题：这些特征是否保存了位置与运动，加入动作后是否能区分不同未来。